# Telugu → Hindi S2S Dataset Builder
**Kaggle T4 · Stream HF → Translate → TTS → Upload HF (single train split)**

In [ ]:
%%capture
!pip install -q git+https://github.com/VarunGumma/IndicTransToolkit
!pip install -q soundfile scipy
import subprocess
result = subprocess.run(
    ["sed", "-i",
     "s/from transformers.tokenization_utils import PreTrainedTokenizerBase/from transformers import PreTrainedTokenizerBase/",
     "/usr/local/lib/python3.12/dist-packages/IndicTransToolkit/collator.py"],
    capture_output=True
)
print("IndicTransToolkit patch:", "ok" if result.returncode == 0 else result.stderr.decode())

In [ ]:
%%capture
!pip uninstall -q -y IndicTransToolkit
!pip install -q "transformers==4.40.0"
!pip install -q git+https://github.com/VarunGumma/IndicTransToolkit
!pip install -q soundfile scipy
!sed -i 's/from transformers.tokenization_utils import PreTrainedTokenizerBase/from transformers import PreTrainedTokenizerBase/' /usr/local/lib/python3.12/dist-packages/IndicTransToolkit/collator.py
!echo "patch:" && grep "PreTrainedTokenizerBase" /usr/local/lib/python3.12/dist-packages/IndicTransToolkit/collator.py

In [ ]:

HF_TOKEN         = "your_huggingface_token_here"

SRC_DATASET      = "ai4bharat/indicvoices"
SRC_CONFIG       = "telugu"
SRC_SPLIT        = "train"

DST_HF_REPO      = "nlpctx/telugu-hindi-s2s"

BATCH_SIZE       = 96
UPLOAD_EVERY     = 500
MAX_SAMPLES      = None

INDICTRANS_MODEL = "ai4bharat/indictrans2-indic-indic-dist-320M"
MMS_TTS_MODEL    = "facebook/mms-tts-hin"

AUDIO_OUT_DIR    = "/kaggle/working/audio_tmp"
CHECKPOINT_FILE  = "/kaggle/working/checkpoint.json"
JSONL_BUFFER     = "/kaggle/working/upload_buffer.jsonl"

import os
os.makedirs(AUDIO_OUT_DIR, exist_ok=True)
print("Config ready.")

In [ ]:
import os, gc, json, shutil, io, warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
import soundfile as sf
from tqdm.auto import tqdm

from huggingface_hub import login, HfApi, create_repo
from datasets import load_dataset, Dataset, Audio, Features, Value, Sequence

login(HF_TOKEN)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

api = HfApi()
try:
    create_repo(DST_HF_REPO, repo_type="dataset", exist_ok=True)
    print(f"HF repo ready: {DST_HF_REPO}")
except Exception as e:
    print(f"Repo: {e}")

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

print("Loading IndicTrans2 …")
indic_tokenizer = AutoTokenizer.from_pretrained(INDICTRANS_MODEL, trust_remote_code=True)
indic_model = AutoModelForSeq2SeqLM.from_pretrained(
    INDICTRANS_MODEL, trust_remote_code=True, torch_dtype=torch.float16
).to(device)
indic_model.eval()
indic_processor = IndicProcessor(inference=True)
print("IndicTrans2 ready.")

In [ ]:
from transformers import VitsModel, AutoTokenizer as TTSTokenizer

print("Loading MMS-TTS Hindi …")
tts_tokenizer = TTSTokenizer.from_pretrained(MMS_TTS_MODEL)
tts_model = VitsModel.from_pretrained(MMS_TTS_MODEL).to(device)
tts_model.eval()
TTS_SR = tts_model.config.sampling_rate
print(f"MMS-TTS ready. SR={TTS_SR}")

In [ ]:

def decode_audio(row):
    try:
        decoder = row.get("audio_filepath", None)
        if decoder is None:
            return None
        decoded = decoder.get_all_samples()
        arr = decoded.data.numpy()
        if arr.ndim == 2:
            arr = arr.mean(axis=0)
        return {"array": arr.astype(np.float32), "sampling_rate": int(decoded.sample_rate)}
    except Exception as e:
        print(f"    decode_audio error: {e}")
        return None

def synthesize_hindi(text):
    try:
        text = text.strip()
        inputs = tts_tokenizer(text, return_tensors="pt", truncation=True, max_length=300).to(device)
        with torch.no_grad():
            waveform = tts_model(**inputs).waveform
        arr = waveform.squeeze().cpu().float().numpy()
        return {"array": arr, "sampling_rate": TTS_SR}
    except Exception as e:
        print(f"    TTS error: {e}")
        return None


def translate_batch(texts, src_lang="tel_Telu", tgt_lang="hin_Deva", sub_batch=32):
    results = []
    for i in range(0, len(texts), sub_batch):
        chunk = [t if t and t.strip() else " " for t in texts[i:i+sub_batch]]
        processed = indic_processor.preprocess_batch(chunk, src_lang=src_lang, tgt_lang=tgt_lang)
        enc = indic_tokenizer(
            processed, truncation=True, padding="longest",
            return_tensors="pt", max_length=256
        ).to(device)
        with torch.no_grad():
            gen = indic_model.generate(**enc, num_beams=4, max_length=256)
        decoded = indic_tokenizer.batch_decode(gen, skip_special_tokens=True)
        results.extend(indic_processor.postprocess_batch(decoded, lang=tgt_lang))
    return results


def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            return json.load(f)
    return {"samples_done": 0, "batches_done": 0}

def save_checkpoint(samples_done, batches_done):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({"samples_done": samples_done, "batches_done": batches_done}, f)

print("Helpers ready.")

In [ ]:

ckpt         = load_checkpoint()
SKIP_SAMPLES = ckpt.get("samples_done", 0)
SKIP_BATCHES = ckpt.get("batches_done", 0)
print(f"Resuming from sample {SKIP_SAMPLES}, batch {SKIP_BATCHES}.")

stream_ds = load_dataset(SRC_DATASET, SRC_CONFIG, split=SRC_SPLIT, streaming=True)

global_idx = 0
batch_num  = SKIP_BATCHES
buffer     = []

KEEP_COLS = [
    "text", "duration", "lang", "samples", "verbatim", "normalized",
    "speaker_id", "scenario", "task_name", "gender", "age_group",
    "job_type", "qualification", "area", "district", "state",
    "occupation", "verification_report"
]

def make_row(r, tel_audio, hin_text, hin_audio):
    row = {col: r.get(col, None) for col in KEEP_COLS}
    row["tel_audio"] = tel_audio
    row["hin_text"]  = hin_text
    row["hin_audio"] = hin_audio
    return row

for row in tqdm(stream_ds, desc="Streaming"):
    global_idx += 1

    if global_idx <= SKIP_SAMPLES:
        continue
    if MAX_SAMPLES and (global_idx - SKIP_SAMPLES) > MAX_SAMPLES:
        print("MAX_SAMPLES reached.")
        break

    buffer.append(row)

    if len(buffer) < BATCH_SIZE:
        continue

    batch_num += 1
    print(f"\n── Batch {batch_num} ({len(buffer)} samples) ──")

    telugu_texts = [r.get("normalized") or r.get("text") or "" for r in buffer]

    print("  Translating …")
    hindi_texts = translate_batch(telugu_texts)

    print("  Synthesising …")
    rows_out = []
    for i, (r, hin_text) in enumerate(zip(buffer, hindi_texts)):
        tel_audio = decode_audio(r)
        hin_audio = synthesize_hindi(hin_text)
        rows_out.append(make_row(r, tel_audio, hin_text, hin_audio))

    print("  Uploading shard …")
    try:
        hf_ds = Dataset.from_list(rows_out)
        hf_ds = hf_ds.cast_column("tel_audio", Audio())
        hf_ds = hf_ds.cast_column("hin_audio", Audio(sampling_rate=TTS_SR))
        shard_path = f"/kaggle/working/shard_{batch_num:05d}.parquet"
        hf_ds.to_parquet(shard_path)
        api.upload_file(
            path_or_fileobj=shard_path,
            path_in_repo=f"data/train-{batch_num:05d}.parquet",
            repo_id=DST_HF_REPO,
            repo_type="dataset",
            token=HF_TOKEN,
        )
        os.remove(shard_path)
        print(f"  ✓ shard {batch_num} uploaded.")
    except Exception as e:
        print(f"  Upload error: {e}")

    save_checkpoint(global_idx, batch_num)

    del hf_ds, rows_out, buffer, hindi_texts, telugu_texts
    gc.collect()
    torch.cuda.empty_cache()
    buffer = []

if buffer:
    batch_num += 1
    print(f"\n── Final partial batch {batch_num} ({len(buffer)} samples) ──")
    telugu_texts = [r.get("normalized") or r.get("text") or "" for r in buffer]
    hindi_texts  = translate_batch(telugu_texts)
    rows_out = []
    for i, (r, hin_text) in enumerate(zip(buffer, hindi_texts)):
        tel_audio = decode_audio(r)
        hin_audio = synthesize_hindi(hin_text)
        rows_out.append(make_row(r, tel_audio, hin_text, hin_audio))
    try:
        hf_ds = Dataset.from_list(rows_out)
        hf_ds = hf_ds.cast_column("tel_audio", Audio())
        hf_ds = hf_ds.cast_column("hin_audio", Audio(sampling_rate=TTS_SR))
        shard_path = f"/kaggle/working/shard_{batch_num:05d}.parquet"
        hf_ds.to_parquet(shard_path)
        api.upload_file(
            path_or_fileobj=shard_path,
            path_in_repo=f"data/train-{batch_num:05d}.parquet",
            repo_id=DST_HF_REPO,
            repo_type="dataset",
            token=HF_TOKEN,
        )
        os.remove(shard_path)
        print(f"  ✓ final shard {batch_num} uploaded.")
    except Exception as e:
        print(f"  Final upload error: {e}")
    save_checkpoint(global_idx, batch_num)

print(f"\n✅ ALL DONE — {batch_num} shards uploaded to {DST_HF_REPO}")

In [ ]:
from datasets import load_dataset as ld
from IPython.display import Audio, display

peek = ld(DST_HF_REPO, split="train", token=HF_TOKEN, streaming=True)
sample = next(iter(peek))
print("tel_text:", sample["tel_text"])
print("hin_text:", sample["hin_text"])
print("tel_audio_sr:", sample["tel_audio_sr"])
if sample.get("hin_audio_path"):
    hin = sample["hin_audio_path"]
    display(Audio(hin["array"], rate=hin["sampling_rate"]))